# Lab 07 solution: let the specialists hand back, and carry a conversation through the manager

This notebook solves the challenge of Lab 07. It repeats the setup it needs from the lab (the config cell, the three
lookups as tools, the two specialists, the triage agent and the manager), so it runs on its own.

## Setup

The same config cell as every lab, then the lab's definitions without their explanations (Parts B and F of the lab
explain each one). Tracing stays in the notebook, as in the lab: `set_trace_processors([])` replaces the uploader with
nothing, since this notebook reads the run results only.

In [1]:
from pathlib import Path
import asyncio
import json
import os
import re
import sqlite3
import time
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
from agents import Agent, ModelSettings, Runner, SQLiteSession, function_tool, set_trace_processors
from openai.types.shared import Reasoning

set_trace_processors([])

DB_PATH = ROOT / "data" / "chinook.db"
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True, check_same_thread=False)


def find_customer(name: str) -> list[dict]:
    """Find a shop customer by full name. Returns a list of matches with their customer_id.

    Args:
        name: First and last name, for example 'Ana Silva'.
    """
    rows = conn.execute(
        "SELECT CustomerId, FirstName || ' ' || LastName, Country FROM customers "
        "WHERE FirstName || ' ' || LastName = ?", (name,)).fetchall()
    return [{"customer_id": r[0], "name": r[1], "country": r[2]} for r in rows]


def get_invoices(customer_id: int) -> dict:
    """Count a customer's invoices and add up their total in US dollars."""
    count, total = conn.execute(
        "SELECT COUNT(*), ROUND(SUM(Total), 2) FROM invoices WHERE CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "invoice_count": count, "total": total}


def get_support_rep(customer_id: int) -> dict:
    """Name the support rep who looks after a customer."""
    row = conn.execute(
        "SELECT e.FirstName || ' ' || e.LastName FROM customers c "
        "JOIN employees e ON e.EmployeeId = c.SupportRepId WHERE c.CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "support_rep": row[0] if row else None}


SETTINGS = ModelSettings(reasoning=Reasoning(effort=EFFORT))
find_customer_tool = function_tool(find_customer)

invoices_agent = Agent(
    name="Invoices",
    handoff_description="Answers questions about a customer's invoices: how many, and their total.",
    instructions=("You answer questions about a customer's invoices. Look up every fact with your tools; "
                  "never answer from memory. Answer in one or two sentences."),
    model=MODEL, tools=[find_customer_tool, function_tool(get_invoices)], model_settings=SETTINGS,
)
reps_agent = Agent(
    name="Support reps",
    handoff_description="Answers who a customer's support rep is.",
    instructions=("You answer who looks after a customer. Look up every fact with your tools; "
                  "never answer from memory. Answer in one or two sentences."),
    model=MODEL, tools=[find_customer_tool, function_tool(get_support_rep)], model_settings=SETTINGS,
)
triage = Agent(
    name="Triage",
    instructions=("You route each request to the right specialist. Hand off to Invoices for invoice counts and "
                  "totals, and to Support reps for who looks after a customer."),
    model=MODEL, handoffs=[invoices_agent, reps_agent], model_settings=SETTINGS,
)
manager = Agent(
    name="Manager",
    instructions=("You help the support reps of a music shop. Ask the specialists for every fact; never answer from "
                  "memory. Answer in one or two sentences."),
    model=MODEL,
    tools=[
        invoices_agent.as_tool(tool_name="ask_invoices",
                               tool_description="Ask the invoices specialist about a customer's invoices: how many, and their total."),
        reps_agent.as_tool(tool_name="ask_support_reps",
                           tool_description="Ask the support-rep specialist who looks after a customer."),
    ],
    model_settings=SETTINGS,
)

PUJA = "How many invoices does Puja Srivastava have, what do they add up to, and who is her support rep?"


def facts_found(text, facts):
    return [bool(re.search(rf"\b{re.escape(fact)}\b", text)) for fact in facts]


print([agent.name for agent in [invoices_agent, reps_agent, triage, manager]])

['Invoices', 'Support reps', 'Triage', 'Manager']


## Task 1: Let the specialists hand back

In the lab the handoff lost the rep because control went to `Invoices` and stayed there. Here each specialist can
hand back to a triage agent, and the triage can hand on to the other specialist.

`clone(...)` copies an agent and changes the fields we name. The three copies refer to each other, so we create the
two specialists first with no handoffs, then the triage with handoffs to both, then set each specialist's `handoffs`
to the new triage.

In [3]:
BACK = (" If the request also asks for something your tools cannot look up, hand off to Triage with what is "
        "still missing.")

invoices_back = invoices_agent.clone(instructions=invoices_agent.instructions + BACK)
reps_back = reps_agent.clone(instructions=reps_agent.instructions + BACK)
triage_back = triage.clone(
    instructions=triage.instructions + " When a specialist hands back, hand off to the specialist for what is "
                                       "still missing.",
    handoffs=[invoices_back, reps_back],
)
invoices_back.handoffs = [triage_back]
reps_back.handoffs = [triage_back]

for agent in [triage_back, invoices_back, reps_back]:
    print(agent.name, "->", [target.name for target in agent.handoffs])

Triage -> ['Invoices', 'Support reps']
Invoices -> ['Triage']
Support reps -> ['Triage']


Every agent can now reach every other through the triage. `max_turns=12` leaves room for several handoffs: each one
is a turn of the agent that hands off.

In [4]:
PUJA_FACTS = ["6", "36.64", "Jane Peacock"]
back_runs = []

for run in range(1, 6):
    result = await Runner.run(triage_back, PUJA, max_turns=12)
    path = [f"{item.source_agent.name} -> {item.target_agent.name}"
            for item in result.new_items if item.type == "handoff_output_item"]
    found = facts_found(result.final_output, PUJA_FACTS)
    back_runs.append(result)
    print(f"run {run} | {' '.join('✅' if f else '❌' for f in found)} | last agent {result.last_agent.name:12} | "
          f"requests {result.context_wrapper.usage.requests} | input tokens {result.context_wrapper.usage.input_tokens}")
    print("        handoffs:  ", " | ".join(path))
    print("        tool calls:", ", ".join(f"{item.agent.name}: {item.raw_item.name}"
                                          for item in result.new_items if item.type == "tool_call_item"))

run 1 | ✅ ✅ ✅ | last agent Support reps | requests 7 | input tokens 2688
        handoffs:   Triage -> Invoices | Invoices -> Triage | Triage -> Support reps
        tool calls: Invoices: find_customer, Invoices: get_invoices, Support reps: get_support_rep


run 2 | ✅ ✅ ✅ | last agent Support reps | requests 7 | input tokens 2688
        handoffs:   Triage -> Invoices | Invoices -> Triage | Triage -> Support reps
        tool calls: Invoices: find_customer, Invoices: get_invoices, Support reps: get_support_rep


run 3 | ✅ ✅ ✅ | last agent Support reps | requests 7 | input tokens 2688
        handoffs:   Triage -> Invoices | Invoices -> Triage | Triage -> Support reps
        tool calls: Invoices: find_customer, Invoices: get_invoices, Support reps: get_support_rep


run 4 | ✅ ✅ ✅ | last agent Support reps | requests 7 | input tokens 2688
        handoffs:   Triage -> Invoices | Invoices -> Triage | Triage -> Support reps
        tool calls: Invoices: find_customer, Invoices: get_invoices, Support reps: get_support_rep


run 5 | ✅ ✅ ✅ | last agent Support reps | requests 8 | input tokens 3298
        handoffs:   Triage -> Invoices | Invoices -> Triage | Triage -> Support reps
        tool calls: Invoices: find_customer, Invoices: get_invoices, Support reps: find_customer, Support reps: get_support_rep


Every message the agents wrote, run by run, with the agent that wrote it (`item.agent`). The last one is the run's
answer:

In [5]:
for run, result in enumerate(back_runs, start=1):
    print(f"run {run}")
    for item in result.new_items:
        if item.type == "message_output_item":
            print(f"    {item.agent.name}: {item.raw_item.content[0].text}")

run 1
    Support reps: Puja Srivastava has 6 invoices totaling 36.64, and her support rep is Jane Peacock.
run 2
    Support reps: Puja Srivastava has 6 invoices totaling $36.64, and her support rep is Jane Peacock.
run 3
    Support reps: Puja Srivastava has 6 invoices totaling $36.64, and her support rep is Jane Peacock.
run 4
    Support reps: Puja Srivastava has 6 invoices totaling $36.64, and her support rep is Jane Peacock.
run 5
    Support reps: Puja Srivastava has 6 invoices totaling $36.64, and her support rep is Jane Peacock.


In this run all five answers had the three facts, at 7 or 8 requests and 2,688 to 3,298 input tokens, and every run took the same path: `Triage -> Invoices -> Triage -> Support reps`, with `Invoices` looking up the invoices and `Support reps` the rep.

Handing back works, but not every time: over the four sets of five runs we made of this cell, 15 of 20 had all three facts. In the misses the run still ended with `Support reps`, after 6 requests (one fewer than a run in which it called `get_support_rep`), and its reply repeated the invoice facts and said *I couldn’t look up her support rep, so I’ve handed that part to Triage*. A handoff gives the agent the whole conversation, and the agent decides for itself what to do with it.

Compared with the lab's five runs of the specialists as tools (all three facts, 8 requests each): handing back made
control come back, but through the conversation, with each agent free to write its own answer. Each handoff is a
model call that hands on, and each agent reads the whole conversation again, so the cost rises with every hop. When
a question needs several specialists and one answer, calling them as tools does the same job with the manager in
charge.

## Task 2: Carry a conversation through the manager

The session belongs to the run of the manager. The specialists run as tools, in nested runs, and see only the `input`
the manager writes for them.

In [6]:
SESSION_FILE = ROOT / "07_OpenAI_Agents_SDK" / "outputs" / "sessions.db"
SESSION_FILE.parent.mkdir(exist_ok=True)
session = SQLiteSession("manager-puja-helena", SESSION_FILE)
await session.clear_session()

first = await Runner.run(manager, PUJA, session=session)
print("run 1:", first.final_output)
second = await Runner.run(manager, "And what about Helena Holý?", session=session)
print("run 2:", second.final_output)
print("Helena's facts:", facts_found(second.final_output, ["7", "49.62", "Steve Johnson"]))
print()
for item in second.new_items:
    if item.type == "tool_call_item":
        print("manager asked:", item.raw_item.name, item.raw_item.arguments)

run 1: Puja Srivastava has 6 invoices totaling $36.64, and her support rep is Jane Peacock.


run 2: Helena Holý has 7 invoices totaling $49.62, and her support rep is Steve Johnson.
Helena's facts: [True, True, True]

manager asked: ask_invoices {"input":"How many invoices does Helena Holý have, and what is their total amount?"}
manager asked: ask_support_reps {"input":"Who is the support rep for Helena Holý?"}


Run 2 has Helena's three facts: 7 invoices, $49.62 and Steve Johnson. The manager read *And what about Helena Holý?* together with the session, which held the Puja question and its answer, and wrote each specialist a complete question: `ask_invoices` got *How many invoices does Helena Holý have, and what is their total amount?*, and `ask_support_reps` got *Who is the support rep for Helena Holý?*

**What this shows:** the session gave the manager the context ("what about" means the same three facts), and the
manager turned that context into complete questions. A specialist called as a tool never reads the conversation, so
the manager's `input` has to carry everything the specialist needs.